<a href="https://www.kaggle.com/code/samratrm/gmm-math-notes?scriptVersionId=341268470" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## From Hard Clustering to Soft Clustering

### The one label problem

K-Means gives every customer exactly one label. Cluster 3, full stop — no degrees, no doubt.

Consider two customers. One sits dead in the middle of the `elite` group: high income, high spend, unmistakable. The other sits on the fence between `elite` and `mature_mainstream`, and could reasonably belong to either.

K-Means labels both of them `elite` and reports nothing to distinguish them. Read the output and they look equally certain. They aren't — and that difference has been permanently destroyed.

This is **hard clustering**. Every point is assigned with 100% confidence, whether or not the confidence is warranted. The algorithm has no vocabulary for doubt.

### What soft clustering gives you back

GMM's output isn't a label. It's a **row of probabilities** — one number per cluster, summing to 1.

Our two customers now look completely different:

| | elite | mature | aspirational | affluent | low-eng |
|---|---|---|---|---|---|
| Customer A | 0.98 | 0.01 | 0.00 | 0.01 | 0.00 |
| Customer B | 0.55 | 0.44 | 0.00 | 0.01 | 0.00 |

Same label if you take the argmax. Completely different business situations. Customer A is a confident `elite` — market to them as one. Customer B is a genuine borderline case, and knowing that is *actionable*: they might be worth targeting with both campaigns, or worth a closer look before you spend on them.

That's the upgrade in one line: **K-Means tells you which group; GMM tells you which group and how sure it is.**

### Clusters stop being points and become clouds

Behind that change is a change in what a "cluster" even is.

For K-Means, a cluster **is** its centroid — a single point in space. That's the entire representation. Everything else follows from it, including two limitations you don't get a choice about:

- Because nearness is measured by plain straight-line distance, every cluster is effectively a **circle**.
- Because all clusters use the same distance rule, every circle is the **same size**.

For GMM, a cluster is a **cloud** with three properties:

- **Centre** — where the segment sits
- **Shape** — how the cloud is stretched and which way it's tilted
- **Weight** — what share of all customers it holds

Each of those unlocks something K-Means structurally cannot do.

**Shape** means a segment can be long and narrow rather than round. Your `affluent_non_spenders` vary a lot in income but barely at all in spending — that's an ellipse, and K-Means can only draw a circle around it, wrongly including people it shouldn't and excluding people it should.

**Weight** means cluster size affects assignment. If one segment holds 5% of customers and another holds 40%, a borderline customer is genuinely more likely to belong to the big one. GMM factors that in. K-Means treats a 10-person cluster and a 90-person cluster as equally plausible claims on every point.

**Both together** mean the boundaries between clusters can **curve**. K-Means boundaries are always straight lines, no exceptions.

### From "which centre is closest" to "what produced this"

The deepest difference is in the question being asked.

K-Means asks a **geometric** question: which centre is nearest? It's carving territory. The output is a partition of space — a map with borders.

GMM asks a **generative** question: what process would produce data that looks like this? It's building a story: customers get created by picking a segment and then drawing from that segment's cloud. Fitting the model means working backwards to figure out what the segments must have been.

Because it commits to a story about how the data arose, GMM ends up with something K-Means never has — a **probability for any point in space**. That gives you two capabilities for free:

- **Anomaly detection.** A customer with low probability under every cloud resembles no segment at all. K-Means has no way to express this; it will cheerfully assign an obvious outlier to its nearest centroid and say nothing.
- **Simulation.** You can generate synthetic customers from the fitted model, because you have an actual distribution, not just a set of borders.

K-Means produces a *partition*. GMM produces a *model*.

### The two are more related than they look

Take a GMM and force every cloud to be a tiny circle of identical size. The probabilities stop being graded — each point becomes essentially certain about its nearest cloud, and every row collapses to a single 1 and the rest 0s.

That's K-Means.

Which reframes the whole comparison: **K-Means isn't a different algorithm, it's GMM with shape, size, and uncertainty switched off.** GMM is the general case; K-Means is the special case that happens to be faster.

### Where GMM stops

Clouds are ellipses. That's a real constraint, not a technicality — an ellipse always has its densest region at its centre.

So picture a ring-shaped cluster. Its centre is *empty*. No ellipse can describe it, and GMM will confidently fit a blob straight through the hole. Same failure for crescents, spirals, and anything else with a curved spine.

It also still needs you to choose the number of clusters, and it's still pulled around by extreme values.

Those are precisely the gaps DBSCAN fills — it makes no assumption about shape at all, finds its own cluster count, and labels outliers as outliers instead of forcing them into a group.


## Expectation–Maximization (EM)

EM is a general recipe for fitting a model when part of the data is **missing or hidden**. In GMM, the hidden thing is which cluster each point came from.

**The problem it solves.** You need the clusters to work out the memberships, and the memberships to work out the clusters. You start with neither, and there's no formula that solves for both at once.

**The trick.** Guess one, then alternate:

**E-step (Expectation)** — Hold the clusters fixed. For each point, work out how likely it is to have come from each cluster. You can't observe the true membership, so you compute its *expected* value: a set of probabilities rather than a single answer.

**M-step (Maximization)** — Hold those memberships fixed. Recompute each cluster's centre, shape, and weight from the points that belong to it — with each point counted only in proportion to how much it belongs. This is the step that *maximizes* how well the model fits.

Repeat until nothing moves.

**Why it's safe.** Every full round is guaranteed to fit the data at least as well as the previous one, never worse. Since the fit can't improve forever, the process must settle.

**Why it isn't perfect.** It settles at a *local* best, not necessarily the global one — different starting guesses can land in different places. That's what `n_init` is for: run it several times from different starts and keep the best.

**Worth knowing:** EM isn't specific to clustering. The same alternating pattern fits Hidden Markov Models, imputes missing values, and trains topic models. GMM is just the cleanest place to learn it, because the hidden variable is so easy to picture.

**And the connection back:** K-Means is EM where the E-step is forced to answer with a hard 0 or 1 instead of a probability.

## The Big Idea: Fuzzy Flashlights

Imagine you have 4 kids standing on a dark number line: **10, 20, 80, and 90**.

You know there are 2 main groups hanging out, but nobody is wearing a label.

* **K-Means** acts like a sharp knife: it draws a line at 50 and says, *"You two on the left are 100% Group A. You two on the right are 100% Group B."*
* **GMM** acts like 2 fuzzy flashlights (bell curves) shining on the line. It says, *"The kid at 10 is 99% in Group A light and 1% in Group B light."*

GMM needs to figure out **where to point the flashlights** (the centers/means), **how wide to open the beam** (the spread/variance), and **how bright each light is** (the cluster weights).

### Step 1: How GMM Uses K-Means to Get Started

GMM can't start blind, so it uses K-Means as a fast, rough guess to set up its starting numbers.

Suppose K-Means splits our points into two groups:

* **Group A:** `[10, 20]`
* **Group B:** `[80, 90]`

GMM looks at these rough groups and sets its initial parameters:

1. **Initial Weights ($\pi$):** How much of the total data belongs to each group?

$$\text{Weight of A} = \frac{2 \text{ points}}{4 \text{ total points}} = 0.5 \text{ (50\%)}$$

$$\text{Weight of B} = \frac{2 \text{ points}}{4 \text{ total points}} = 0.5 \text{ (50\%)}$$

2. **Initial Means ($\mu$):** Where is the center of each group?

$$\text{Center of A} = \frac{10 + 20}{2} = 15$$

$$\text{Center of B} = \frac{80 + 90}{2} = 85$$

3. **Initial Spread ($\sigma^2$):** How wide is each group?

* Group A points (10 and 20) sit 5 units away from center 15.
* Average squared distance = $\frac{5^2 + 5^2}{2} = 25$.

Now GMM has its starting flashlights:

* Flashlight A is centered at **15**, has a spread of **25**, and accounts for **50%** of the team.
* Flashlight B is centered at **85**, has a spread of **25**, and accounts for **50%** of the team.


### Step 2: The Update Loop (The EM Algorithm)

Now GMM throws away K-Means' hard borders and starts tuning its parameters using a two-step cycle called **Expectation-Maximization (EM)**.

**Part A: The E-Step (Handing out "Ownership Tickets")**

Every point gets 1 full ticket to split between Group A and Group B based on how close it is to each flashlight beam.

* **Point 10:** Super close to Center 15, miles away from 85.
* Ticket split: **0.99 for A**, **0.01 for B**.


* **Point 20:** Close to Center 15, far from 85.
* Ticket split: **0.95 for A**, **0.05 for B**.


* **Point 80:**
* Ticket split: **0.05 for A**, **0.95 for B**.


* **Point 90:**
* Ticket split: **0.01 for A**, **0.99 for B**.



**Part B: The M-Step (Updating the Parameters)**

Now we recalculate the flashlights using these fractional tickets.

1. **Update the Weights:**
Add up all ticket fractions assigned to Group A:

$$\text{Total A tickets} = 0.99 + 0.95 + 0.05 + 0.01 = 2.0$$

$$\text{New Weight for A} = \frac{2.0 \text{ tickets}}{4 \text{ total kids}} = 0.50 \text{ (50\%)}$$

2. **Update the Mean (Center):**
Every point pulls on the center, but points with bigger ticket shares pull harder:

$$\text{New Center for A} = \frac{(0.99 \times 10) + (0.95 \times 20) + (0.05 \times 80) + (0.01 \times 90)}{2.0 \text{ total tickets}}$$

$$\text{New Center for A} = \frac{9.9 + 19.0 + 4.0 + 0.9}{2.0} = \frac{33.8}{2.0} = 16.9$$

Notice how Point 80 (which is mostly in Group B) only pulled Center A by a tiny fraction ($0.05 \times 80 = 4.0$), whereas Point 10 and 20 did almost all the pulling!

3. **Update the Spread:**
We measure how far points are from the *new center* (16.9), weighted by their ticket values:

$$\text{New Spread} = \frac{0.99 \times (10 - 16.9)^2 + 0.95 \times (20 - 16.9)^2 + \dots}{2.0 \text{ total tickets}}$$


### Step 3: How Do We Get the Final Answer?

We repeat the **E-step** and **M-step** back and forth:

1. **E-step:** Recalculate ticket splits using the new centers and spreads.
2. **M-step:** Move the centers and adjust spreads based on the new ticket splits.

**When does it stop?**
During the first few loops, the centers might move from $15.0 \rightarrow 16.9 \rightarrow 15.2 \rightarrow 15.01$.

Eventually, the updates become infinitesimally small (e.g., center moves by less than $0.00001$). This is called **convergence**. The algorithm freezes and outputs its final result:

* **Final Parameters:** The exact center, spread, and weight of each flashlight cloud.
* **Final Probabilities:** For every single customer, a row of probabilities showing exactly how confident the model is about their segment membership.
----

# Gaussian Mixture Models: Math Intuition

## Part 0: The one idea that makes everything else click

K-Means asks a **geometric** question: *which centroid is nearest?*

GMM asks a **generative** question: *what random process could have produced this data?*

That change of question is the whole subject. Everything below is a consequence of it.

### The generative story

GMM assumes your 200 mall customers were produced like this:

1. Nature rolls a weighted die with $K$ faces and picks a cluster $k$.
2. Nature draws a point from that cluster's Gaussian bell curve.
3. You are shown the point — **but not which face the die landed on**.

That hidden die roll is called a **latent variable**, written $z_i$ for customer $i$. You never observe it.

So the task is inversion: *given the points, work backwards to the die weights and the bell curves.* And here is the circularity that defines the problem:

- If you knew the cluster assignments, fitting each Gaussian would be trivial arithmetic.
- If you knew the Gaussians, computing the assignments would be trivial arithmetic.
- You know neither.

EM breaks that circle by alternating between the two. That's it. That's the algorithm.

## Part 1: One Gaussian in 1D

Before mixing several, understand one.

$$p(x \mid \mu, \sigma^2) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

Variables:

- $x$ — the value you observe (e.g. spending score = 45)
- $\mu$ — the mean, the centre of the bell
- $\sigma^2$ — the variance, how wide the bell is
- $\sigma$ — the standard deviation, $\sqrt{\text{variance}}$

Read it in two halves.

**The exponent** $-\frac{(x-\mu)^2}{2\sigma^2}$ is the interesting part. Notice $(x-\mu)^2$ is **squared distance from the centre**, and dividing by $\sigma^2$ measures that distance *in units of the cluster's own width*. Being 20 units from the centre is far if $\sigma = 5$ and close if $\sigma = 50$. This is the single most important idea in the formula: **distance is relative to spread.** K-Means has no equivalent.

**The prefactor** $\frac{1}{\sigma\sqrt{2\pi}}$ just forces the total area to equal 1, so it's a valid probability distribution. It looks like bookkeeping, and you can often ignore it — but *not* when two clusters have different $\sigma$, because then it differs between them. You'll see this bite in Example 3.

### Example 1: reading a density

Let $\mu = 50$, $\sigma = 10$, and $x = 60$.

$$\frac{(60-50)^2}{2 \cdot 10^2} = \frac{100}{200} = 0.5$$

So the exponential part is $e^{-0.5} = 0.607$. The point sits exactly one standard deviation out, and its density is about 61% of the peak density.

Try $x = 70$ (two $\sigma$ out): exponent $= 400/200 = 2$, giving $e^{-2} = 0.135$. **Twice as far, but density fell to a fifth**, not a half — because distance enters squared and then gets exponentiated. Gaussians drop off *fast*. This is why GMM, like K-Means, is sensitive to outliers.

## Part 2: Mixing them

A mixture is a weighted sum of Gaussians:

$$p(x) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(x \mid \mu_k, \Sigma_k)$$

Variables:

- $K$ — number of components (your `n_components`)
- $\pi_k$ — the **mixing weight** of component $k$: the probability the die lands on face $k$
- $\mu_k$ — mean vector of component $k$
- $\Sigma_k$ — covariance matrix of component $k$ (capital sigma; Part 5)
- $\mathcal{N}(\cdot)$ — the Gaussian density from Part 1

    **Note:**
    
    The initialization of $\pi_k$ isn't random by default, and $\pi_k$ isn't something you *give*, it's something the model *learns*.
    
    **Initialization.** sklearn's default is `init_params='kmeans'`: it runs K-Means first, then sets $\pi_k$ to the fraction of points each K-Means cluster got. So if K-Means put 40 of 200 points in cluster 1, initial $\pi_1 = 0.2$. Other options are `'random'`, `'k-means++'`, `'random_from_data'`.
    
    **Then EM updates it.** 


Two constraints on the weights, both from them being probabilities of a die roll:

$$\pi_k \geq 0 \qquad \text{and} \qquad \sum_{k=1}^{K}\pi_k = 1$$

In sklearn these are `gm.weights_`, `gm.means_`, `gm.covariances_`. The whole model is those three arrays.

### Example 2: a two-component mixture

Suppose after fitting you get:

| Component | $\pi_k$ | $\mu_k$ | $\sigma_k$ | Interpretation |
|---|---|---|---|---|
| A | 0.6 | 25 | 8 | low spenders, 60% of base |
| B | 0.4 | 75 | 12 | high spenders, 40% of base |

The density at $x = 50$ is:

$$p(50) = 0.6 \cdot \mathcal{N}(50 \mid 25, 8^2) + 0.4 \cdot \mathcal{N}(50 \mid 75, 12^2)$$

> Both terms are small — 50 is far from both centres in units of their own $\sigma$. That is exactly what a **low-density valley between clusters** looks like numerically. K-Means would confidently assign $x = 50$ to whichever centroid is closer and tell you nothing about the fact that it fits *neither* group well. GMM's density value carries that information, which is what makes anomaly detection possible with `gm.score_samples()`.

## Part 3: Responsibility — the E-step

Now the key quantity. For each point and each component, define the **responsibility**:

$$\gamma_{ik} = \frac{\pi_k \, \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j \, \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}$$

Variables:

- $\gamma_{ik}$ — probability that point $i$ came from component $k$; "gamma"
- the numerator — how well component $k$ explains this point, scaled by how common $k$ is
- the denominator — the same summed over all components, so the $\gamma$ values sum to 1

**This is Bayes' theorem.** Nothing more.

$$\text{posterior} = \frac{\text{prior} \times \text{likelihood}}{\text{evidence}}$$

- **prior** $\pi_k$ — before looking at the point, how likely is cluster $k$? (Big clusters are a priori more likely.)
- **likelihood** $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$ — how well does cluster $k$'s shape explain this point?
- **posterior** $\gamma_{ik}$ — after seeing the point, how likely is cluster $k$?

For each point $i$: $\sum_k \gamma_{ik} = 1$. That row of numbers is exactly what `predict_proba` returns, and `predict` is just its argmax.

### Example 3: three numeric cases, and what each teaches

Setup: two 1D components. A has $\mu_A = 30$, $\sigma_A = 10$. B has $\mu_B = 70$, $\sigma_B = 10$. Our point is $x = 45$.

**Case (a): equal weights, equal widths.** $\pi_A = \pi_B = 0.5$.

Since $\sigma$ is the same for both, the prefactors cancel in the ratio and I can use just the exponentials:

$$\mathcal{N}(45 \mid 30, 10^2) \propto \exp\left(-\frac{15^2}{200}\right) = e^{-1.125} = 0.3247$$
$$\mathcal{N}(45 \mid 70, 10^2) \propto \exp\left(-\frac{25^2}{200}\right) = e^{-3.125} = 0.0439$$

$$\gamma_A = \frac{0.5 \times 0.3247}{0.5 \times 0.3247 + 0.5 \times 0.0439} = \frac{0.3247}{0.3686} = \mathbf{0.881}$$

So $\gamma_B = 0.119$. The point is 15 from A and 25 from B, and GMM says 88% / 12%. **K-Means would say 100% / 0%.** That difference is the soft assignment.

**Case (b): change only the weights.** Now $\pi_A = 0.9$, $\pi_B = 0.1$ — cluster A is nine times more common.

$$\gamma_A = \frac{0.9 \times 0.3247}{0.9 \times 0.3247 + 0.1 \times 0.0439} = \frac{0.2922}{0.2966} = \mathbf{0.985}$$

Same point, same distances, but the answer moved from 88% to 98.5%. **Cluster size affects assignment.** This is a real capability K-Means lacks entirely: a rare cluster has to work harder to claim a point.

**Case (c): change only the widths.** Back to $\pi_A = \pi_B = 0.5$, but let $\sigma_B = 30$ — B is a broad, diffuse cluster.

Now the prefactors *don't* cancel, so compute full densities:

$$\mathcal{N}(45 \mid 30, 10^2) = \frac{1}{10\sqrt{2\pi}}e^{-1.125} = 0.03989 \times 0.3247 = 0.01295$$
$$\mathcal{N}(45 \mid 70, 30^2) = \frac{1}{30\sqrt{2\pi}}e^{-0.347} = 0.01330 \times 0.7066 = 0.00940$$

$$\gamma_A = \frac{0.01295}{0.01295 + 0.00940} = \mathbf{0.579}$$

The point dropped from 88% A to 58% A **without moving.** A wide cluster can plausibly claim distant points, because for it, 25 units isn't far. This is the flexibility that `covariance_type` controls, and it's why the choice matters.

Notice also the tension in the two factors: B's exponential got *better* ($e^{-0.347}$ vs $e^{-3.125}$) but its prefactor got *worse* ($1/30$ vs $1/10$). A wide Gaussian spreads its fixed unit of probability mass thinner everywhere. That trade-off is what stops GMM from just making every cluster infinitely wide.

## Part 4: The M-step

The E-step gave every point a fractional membership in every cluster. Now refit the Gaussians using those fractions as weights.

First, the **effective count** — how many points does cluster $k$ own, counting fractions?

$$N_k = \sum_{i=1}^{n} \gamma_{ik}$$

Then the three updates:

$$\pi_k = \frac{N_k}{n} \qquad\qquad \mu_k = \frac{1}{N_k}\sum_{i=1}^{n} \gamma_{ik}\, x_i$$

$$\Sigma_k = \frac{1}{N_k}\sum_{i=1}^{n} \gamma_{ik}\,(x_i - \mu_k)(x_i - \mu_k)^{\top}$$

Variables:

- $n$ — total number of points (200 for you)
- $N_k$ — effective count for cluster $k$; note $\sum_k N_k = n$
- $\top$ — transpose, which turns the outer product into a $d \times d$ matrix

**Read these as the formulas you already know.** $\mu_k$ is just an average. $\Sigma_k$ is just a variance. $\pi_k$ is just a proportion. The *only* modification is that each point is counted $\gamma_{ik}$ times instead of once. Set every $\gamma$ to 0 or 1 and these collapse into the ordinary mean and covariance of a hard-assigned group — which is precisely K-Means' centroid update.

### Example 4: an M-step by hand

Three points: $x = 10, 20, 60$. Suppose the E-step gave cluster A the responsibilities $0.9, 0.8, 0.1$.

Effective count:

$$N_A = 0.9 + 0.8 + 0.1 = 1.8$$

Weighted mean:

$$\mu_A = \frac{0.9(10) + 0.8(20) + 0.1(60)}{1.8} = \frac{9 + 16 + 6}{1.8} = \frac{31}{1.8} = 17.22$$

Weighted variance:

$$\sigma_A^2 = \frac{0.9(10-17.22)^2 + 0.8(20-17.22)^2 + 0.1(60-17.22)^2}{1.8}$$
$$= \frac{0.9(52.2) + 0.8(7.7) + 0.1(1830.0)}{1.8} = \frac{46.9 + 6.2 + 183.0}{1.8} = 131.2$$

So $\sigma_A = 11.45$.

**Look at the contributions: 46.9, 6.2, 183.0.** The point at 60 was given only 10% responsibility, yet it supplies 77% of the variance. Squared distance is brutal. Two lessons fall out of this one calculation:

1. GMM is **not** robust to outliers, despite soft assignment. A far point with tiny responsibility still inflates a covariance badly.
2. This is why `covariance_type='full'` can go wrong on small data — a component can stretch into a thin sliver chasing a few stray points, and in the extreme collapse onto a single point with near-zero variance and infinite likelihood. That's the degenerate solution `reg_covar` exists to prevent.

## Part 5: Covariance, and what the four types mean

For $d$ features, $\Sigma_k$ is a $d \times d$ matrix. With two features (income, spending):

$$\Sigma_k = \begin{bmatrix} \mathrm{Var}(\text{inc}) & \mathrm{Cov}(\text{inc},\text{spd}) \\ \mathrm{Cov}(\text{spd},\text{inc}) & \mathrm{Var}(\text{spd}) \end{bmatrix}$$

- **Diagonal entries** — spread along each axis. Sets the ellipse's width and height.
- **Off-diagonal entries** — how the two features co-vary *inside this cluster*. Sets the ellipse's **tilt**. Zero means axis-aligned.
- The matrix is always symmetric, so $\mathrm{Cov}(a,b) = \mathrm{Cov}(b,a)$. Only $d(d+1)/2$ of its $d^2$ entries are free.

Geometrically, a Gaussian's contours of equal density are ellipses, and $\Sigma$ *is* the ellipse:

```
spherical            diag                 full
  (circle)      (axis-aligned ellipse)   (tilted ellipse)

     ***              *********              ****
   *******           ***********           *******
  *********          ***********          *********
   *******           ***********          *******
     ***              *********           ****

  σ² only          Var(x), Var(y)      Var(x), Var(y), Cov(x,y)
```

The four `covariance_type` settings are increasingly severe restrictions on that matrix:

| Type | Constraint | Free params per component | Shape |
|---|---|---|---|
| `full` | none | $d(d+1)/2$ | any ellipse, any tilt, each different |
| `tied` | one matrix shared by all | $d(d+1)/2$ **total** | identical ellipse for every cluster |
| `diag` | off-diagonals $= 0$ | $d$ | axis-aligned ellipses |
| `spherical` | one scalar $\sigma_k^2$ | $1$ | circles, radii may differ |

### Parameter counts, concretely

Total free parameters $p$ for $K$ components and $d$ features:

$$p = \underbrace{(K-1)}_{\text{weights}} + \underbrace{Kd}_{\text{means}} + \underbrace{p_{\text{cov}}}_{\text{covariances}}$$

Weights get $K-1$ rather than $K$ because they must sum to 1 — the last one is determined.

With your data ($d = 4$, $K = 5$):

| Type | $p_{\text{cov}}$ | Total $p$ |
|---|---|---|
| `spherical` | $K = 5$ | 29 |
| `diag` | $Kd = 20$ | 44 |
| `tied` | $d(d+1)/2 = 10$ | 34 |
| `full` | $K \cdot d(d+1)/2 = 50$ | 74 |

**74 parameters estimated from 200 points.** Under four data points per parameter. Keep that ratio in mind for the next part.

## Part 6: What EM is actually optimizing

The objective is the **log-likelihood** — how probable is the observed data under the model?

$$\ell(\theta) = \sum_{i=1}^{n} \log\left(\sum_{k=1}^{K} \pi_k \,\mathcal{N}(x_i \mid \mu_k, \Sigma_k)\right)$$

where $\theta$ is shorthand for all parameters $\{\pi_k, \mu_k, \Sigma_k\}$. In sklearn, `gm.score(X)` returns $\ell/n$ (the mean, not the sum — a common gotcha).

**Why this is hard:** the $\log$ sits outside a $\sum$. A log of a sum has no clean derivative you can set to zero, so there's no closed-form solution.

**Why EM helps:** if you knew each point's true cluster $z_i$, the inner sum would collapse to a single term, the log would land directly on the Gaussian, and $\log$ of an exponential is just the exponent — an easy quadratic to maximize. The intractability comes *entirely* from not knowing $z_i$.

So EM sidesteps it:

```
   initialize π, μ, Σ   (sklearn defaults to k-means for this)
            │
            ▼
   ┌──────  E-step  ─────────────────────┐
   │  fix the Gaussians                  │
   │  compute every γ_ik via Bayes       │
   └──────────────┬──────────────────────┘
                  ▼
   ┌──────  M-step  ─────────────────────┐
   │  fix the responsibilities           │
   │  recompute π, μ, Σ as weighted stats│
   └──────────────┬──────────────────────┘
                  ▼
      has ℓ stopped increasing?
         no ──► back to E-step
         yes ─► converged
```

**The guarantee:** each full iteration provably never decreases $\ell$. Since it's bounded above and monotonically increasing, it must converge.

**The catch:** it converges to a *local* maximum. Different initializations reach different answers — which is exactly why `n_init` exists and why you should set it above 1.

## Part 7: BIC and AIC — paying for complexity

More parameters always fit the training data better, so you can't select $K$ or `covariance_type` by likelihood alone. Both criteria add a penalty:

$$\mathrm{BIC} = -2\ell + p\log n \qquad\qquad \mathrm{AIC} = -2\ell + 2p$$

Variables:

- $\ell$ — the maximized log-likelihood (bigger = better fit)
- $p$ — number of free parameters (from Part 5)
- $n$ — number of data points

The $-2\ell$ term rewards fit; the second term punishes complexity. **Lower is better for both.**

The only difference is the price per parameter: $\log n$ for BIC, a flat $2$ for AIC. Since $\log n > 2$ whenever $n > 7$, **BIC always penalizes more heavily than AIC**, and increasingly so as data grows. BIC prefers simpler models; AIC is more permissive.

### Example 5: the penalty on your data

$n = 200$, so $\log 200 = 5.30$.

| Type | $p$ | BIC penalty $= p \log n$ |
|---|---|---|
| `spherical` | 29 | 154 |
| `tied` | 34 | 180 |
| `diag` | 44 | 233 |
| `full` | 74 | 392 |

`full` starts **159 BIC points behind** `diag`. To win, it must improve $-2\ell$ by more than 159 — that is, raise the log-likelihood by about 80. On 200 rows with four features, that is a lot to ask.

So if your sweep says `diag` or `tied` beats `full`, **that is not a bug and not a disappointment.** It's BIC telling you your dataset is too small to support 50 covariance parameters. The reusable principle: **model flexibility must be paid for in data.** Same logic as regularization; BIC just makes the invoice explicit.

## Part 8: The exact relationship to K-Means

You can now state this precisely. K-Means is the limiting case of GMM under three restrictions:

1. `covariance_type='spherical'` — circular clusters
2. all $\sigma_k^2$ equal, and $\sigma^2 \to 0$
3. equal weights, $\pi_k = 1/K$

Restriction 2 is the interesting one. Watch what happens to a responsibility ratio as $\sigma^2$ shrinks:

$$\frac{\gamma_{iA}}{\gamma_{iB}} = \exp\left(\frac{d_B^2 - d_A^2}{2\sigma^2}\right)$$

where $d_A, d_B$ are distances from point $i$ to each centre. As $\sigma^2 \to 0$, the denominator vanishes, the exponent blows up to $\pm\infty$, and the ratio goes to $\infty$ or $0$. Every responsibility snaps to exactly 1 or 0.

**Soft assignment becomes hard assignment.** Winner-take-all, nearest centroid, K-Means.

Reading it the other way: K-Means is GMM that has thrown away shape, size, and uncertainty. The three things GMM gives you back are exactly those three.

## Part 9: What this buys you on the mall data

| Capability | K-Means | GMM |
|---|---|---|
| Elliptical / tilted clusters | no | yes (`full`, `diag`) |
| Different cluster sizes | no | yes (via $\pi_k$) |
| Membership uncertainty | no | yes (`predict_proba`) |
| Principled model selection | elbow heuristic | BIC / AIC minimum |
| Density estimate per point | no | yes (`score_samples`) |

The business payoff is the third row. `predict_proba` finds your **boundary customers** — someone at 55% `young_aspirational` / 45% `elite` is a person whose segment is genuinely ambiguous, and that's a marketing decision, not a rounding error. K-Means silently rounded them off.

The fifth row is the second payoff: low `score_samples` means "this customer resembles no segment well" — outlier detection for free.

## Part 10: Check yourself

Do these on paper before touching sklearn. If you can't, reread the relevant Part.

**Conceptual**

1. Why is $z_i$ called *latent*, and why does its absence make the log-likelihood hard to maximize?
2. In the responsibility formula, which factor is the prior and which is the likelihood? What does each contribute?
3. Explain why EM can never make the log-likelihood worse, but still needs `n_init > 1`.
4. Why does a wide Gaussian have a *smaller* prefactor? What would break if it didn't?

**Numeric — do the arithmetic**

5. Two components, $\pi_A = \pi_B = 0.5$, $\mu_A = 20$, $\mu_B = 60$, both $\sigma = 15$. Compute $\gamma_A$ for $x = 35$.
6. Repeat with $\pi_A = 0.8, \pi_B = 0.2$. By how much did $\gamma_A$ move, and why?
7. Points $x = 5, 15, 25$ with responsibilities $0.7, 0.6, 0.2$ for cluster A. Compute $N_A$, $\mu_A$, $\sigma_A^2$.
8. For $d = 2$, $K = 3$: compute total $p$ for all four covariance types. Which would BIC favour at $n = 150$?

**From scratch, in numpy — no sklearn, no docs**

9. Write a function `gaussian_1d(x, mu, sigma)` returning the density. Vectorize over `x`.
10. Write `e_step(X, pi, mu, sigma)` returning an $n \times K$ responsibility matrix. Assert each row sums to 1.
11. Write `m_step(X, gamma)` returning updated `pi, mu, sigma`.
12. Loop them into a working 1D GMM. Generate data from two known Gaussians, fit, and check you recover the parameters. Print $\ell$ each iteration and **verify it never decreases** — that's your correctness test.

Exercise 12 is the one that matters. Getting a monotonically increasing log-likelihood out of code you wrote yourself is the moment GMM stops being an API and becomes something you understand.

## What K-Means Cannot Do (and GMM Solves)

* **Elliptical and Oriented Clusters:** Handles arbitrary spatial orientations and feature variances via component covariance matrices ($\Sigma_k$), whereas K-Means assumes isotropic (spherical) clusters with equal variance.
* **Soft (Probabilistic) Assignment:** Yields continuous posterior membership probabilities ($p(z_k \mid x)$ via `predict_proba`), explicitly quantifying classification uncertainty near cluster boundaries rather than forcing hard assignments.
* **Unequal Cluster Weights:** Incorporates component priors ($\pi_k$), preventing small or rare clusters from incorrectly absorbing data points in dense overlap regions.
* **Principled Model Selection:** Evaluates model fit using statistical information criteria (BIC/AIC) based on log-likelihood penalties rather than relying on heuristic geometric checks (such as the elbow method).
* **Generative Density Estimation:** Estimates the true joint probability distribution ($p(x)$ via `score_samples`), enabling probabilistic anomaly detection and synthetic sample generation (`gm.sample()`).
* **Curved Decision Boundaries:** Generates non-linear, quadratic decision boundaries when component covariances differ, replacing the linear Voronoi tessellations produced by K-Means.

## Limitations of Gaussian Mixture Models

* **Fixed Component Count ($K$):** Requires pre-specifying the number of mixture components prior to model fitting.
* **Parametric Shape Constraints:** Strictly bounded by Gaussian geometry; fails on complex non-convex manifold topologies (e.g., spirals, concentric rings, arbitrary density chains) where algorithms like DBSCAN are required.
* **Sensitivity to Outliers:** Relies on squared exponential distances, making parameters highly susceptible to distortion from extreme values.
* **Local Optima Sensitivity:** Expectation-Maximization (EM) only guarantees convergence to local log-likelihood maxima, necessitating multiple initialization restarts (`n_init > 1`).
* **Parameter Inflation & Overfitting:** Estimating covariance matrices requires estimating $\mathcal{O}(K \cdot d^2)$ parameters, demanding significantly larger sample sizes to avoid overfitting.
* **Variance Collapse (Singularities):** A Gaussian component can collapse onto a subset of closely spaced data points, causing variance to approach zero and likelihood to approach infinity (mitigated by setting `reg_covar`).
* **Computational Scale:** Higher runtime complexity and memory footprint than distance-based partitioning, degrading in high-dimensional spaces where covariance matrices become ill-conditioned.

**Core Takeaway:** K-Means strictly partitions geometric space; GMM models the underlying generative probability distribution that produced the data.